# Update Binary Scenario Variables from Google Drive

## Purpose
Syncs exogenous scenario variables from Google Sheets to the `gold.exogenous_variables` table. Detects new scenarios and changed event parameters (direction, type, dates), generates binary timeline columns for event windows, and intelligently updates only modified rows.

## Data Source
* **Google Sheets**: Scenario definitions with event parameters
  * Country, Event Direction (Production/Consumption), Description
  * Start/End Year and Month for event windows
  * Event Type (optional), Coefficient (optional)

## Key Features
* **Change detection**: Identifies new scenarios AND changes to existing scenarios
  * Compares metadata (Event Direction, Event Type)
  * Deep check: Regenerates binary timelines and compares against existing table to detect date changes
* **Binary timeline generation**: Creates monthly binary (0/1) columns indicating when events are active
* **Composite key matching**: Uses Country + Description + Event Direction to handle scenarios that affect both production and consumption
* **Smart updates**: Only processes new or changed rows, preserving unchanged data

## Output
* **Table**: `workspace.gold.exogenous_variables`
  * Metadata columns: Country, Event_Direction, Description, Event_Type, Coefficient, generic_name
  * Binary timeline columns: One column per month (YYYY-MM-DD) with 1 when event is active, 0 otherwise

## Workflow
1. Load Google Sheet data and existing `gold.exogenous_variables` table
2. Identify completely new scenarios (Country+Description+Event_Direction not in table)
3. Check existing scenarios for metadata changes (Event Direction, Event Type)
4. Deep check: For unchanged metadata, regenerate binary timelines and compare to detect date changes
5. Process new and changed rows: generate binary timelines and update table
6. Report summary of changes

In [0]:
# import libraries
import pandas as pd
import sklearn
import numpy as np

In [0]:
# https://docs.google.com/spreadsheets/d/1lf7Qd0MjLfp8C7h5HZuPlOXIvYdlDHUyUYsMUOPoufw/edit?gid=1931808656#gid=1931808656

sheet_id = "1lf7Qd0MjLfp8C7h5HZuPlOXIvYdlDHUyUYsMUOPoufw"
gid_p = "1931808656" # sheet tab ID

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid_p}"

# Load CSV
df = spark.createDataFrame(pd.read_csv(url))

# Convert to pandas for easier manipulation
df_pd = df.toPandas()

In [0]:
# Load the existing exogenous_variables table
existing_table = spark.table('workspace.gold.exogenous_variables')

# Get just the metadata columns for comparison (excluding generic_name which is auto-assigned)
existing_metadata = existing_table.select('Country', 'Event_Direction', 'Description').toPandas()

print(f"Existing table has {existing_table.count()} rows")
print(f"Google Sheet has {len(df_pd)} rows")

display(df_pd.head())

Existing table has 63 rows
Google Sheet has 43 rows


Start Year,Start Month,End Year,End Month,Country,Event Direction,Event Type,Description,Notes
1968,6,null,null,Canada,Production,New Pipeline,Original Enbridge Line 3 Pipeline completed,"Only year of opening given, month is averaged"
1969,6,null,null,Canada,Consumption,New Pipeline,Enbridge Line 6 Pipeline completed,"Only year of opening given, month is averaged"
1973,10,1974.0,3.0,Canada,Consumption,Export Cut,1973 OPEC Oil Embargo,null
1997,6,null,null,Canada,Production,New Pipeline,Express Pipeline completed to Casper,"Only year of opening given, month is averaged"
2010,4,null,null,Canada,Production,New Pipeline,Alberta Clipper Pipeline (Enbridge Line 67) completed,null


In [0]:
# Identify new rows AND changed rows
# Strategy: For existing rows, regenerate binary timeline and compare against table

# Create a composite key for comparison (MUST include Event_Direction since events can be both Production and Consumption)
df_pd['composite_key'] = df_pd['Country'] + '||' + df_pd['Description'] + '||' + df_pd['Event Direction']

# Load full existing table with date columns for comparison
existing_full = existing_table.toPandas()
existing_full['composite_key'] = existing_full['Country'] + '||' + existing_full['Description'] + '||' + existing_full['Event_Direction']

# Find completely NEW rows (Country+Description not in existing table)
new_rows_mask = ~df_pd['composite_key'].isin(existing_full['composite_key'])
new_rows = df_pd[new_rows_mask].copy()

# For existing rows, check if anything changed by regenerating their binary timeline
# and comparing key metadata fields
print("\nChecking existing rows for changes...")
changed_rows = []

for idx, row in df_pd[~new_rows_mask].iterrows():
    composite_key = row['composite_key']
    existing_row = existing_full[existing_full['composite_key'] == composite_key].iloc[0]
    
    # Compare metadata that affects the binary timeline:
    # 1. Event_Direction
    # 2. Event_Type
    # We can't compare dates directly (not stored), but Event_Direction change is a good proxy
    # For comprehensive detection, we'd need to regenerate and compare binary columns
    
    event_dir_changed = row['Event Direction'] != existing_row['Event_Direction']
    event_type_changed = (pd.isna(row.get('Event Type')) != pd.isna(existing_row['Event_Type'])) or \
                        (not pd.isna(row.get('Event Type')) and row.get('Event Type') != existing_row['Event_Type'])
    
    if event_dir_changed or event_type_changed:
        changed_rows.append(idx)

changed_rows_df = df_pd.loc[changed_rows].copy() if changed_rows else pd.DataFrame()

# Drop the composite key column
df_pd = df_pd.drop(columns=['composite_key'])
existing_full = existing_full.drop(columns=['composite_key'])
if len(new_rows) > 0:
    new_rows = new_rows.drop(columns=['composite_key'])
if len(changed_rows_df) > 0:
    changed_rows_df = changed_rows_df.drop(columns=['composite_key'])

print(f"\n{'='*60}")
print(f"CHANGE DETECTION RESULTS")
print(f"{'='*60}")
print(f"New rows to add: {len(new_rows)}")
print(f"Existing rows with changes: {len(changed_rows_df)}")
print(f"{'='*60}\n")

if len(new_rows) > 0:
    print("NEW ROWS:")
    display(new_rows)
    
if len(changed_rows_df) > 0:
    print("\nCHANGED ROWS (will be updated):")
    display(changed_rows_df)

if len(new_rows) == 0 and len(changed_rows_df) == 0:
    print("✓ No obvious changes detected. Running deep check next...")


Checking existing rows for changes...

CHANGE DETECTION RESULTS
New rows to add: 0
Existing rows with changes: 0

✓ No obvious changes detected. Running deep check next...


In [0]:
# Additional check: For rows where metadata looks the same, verify binary timeline hasn't changed
# This catches cases where only Start/End dates changed in Google Sheet

if len(df_pd) - len(new_rows) - len(changed_rows_df) > 0:
    print(f"\nPerforming deep check on {len(df_pd) - len(new_rows) - len(changed_rows_df)} rows with matching metadata...")
    
    # Get date columns from existing table (only compare columns that actually exist)
    metadata_cols_set = set(['Country', 'Event_Direction', 'Event_Type', 'Description', 'Coefficient', 'generic_name'])
    date_columns = [col for col in existing_full.columns if col not in metadata_cols_set]
    
    additional_changed = []
    
    for idx, row in df_pd.iterrows():
        composite_key = row['Country'] + '||' + row['Description'] + '||' + row['Event Direction']
        
        # Skip if already marked as new or changed
        if idx in new_rows.index or idx in changed_rows_df.index:
            continue
            
        # Get existing row
        existing_row = existing_full[existing_full['Country'] + '||' + existing_full['Description'] + '||' + existing_full['Event_Direction'] == composite_key].iloc[0]
        
        # Regenerate binary timeline for this row based on Google Sheet dates
        start_date = pd.Timestamp(year=int(row['Start Year']), month=int(row['Start Month']), day=1)
        if pd.isna(row['End Year']) or pd.isna(row['End Month']):
            end_date = pd.Timestamp(year=int(row['Start Year']), month=12, day=1)
        else:
            end_date = pd.Timestamp(year=int(row['End Year']), month=int(row['End Month']), day=1)
        
        # Compare binary values for each date
        timeline_changed = False
        for date_str in date_columns:
            date = pd.Timestamp(date_str)
            expected_value = 1 if (start_date <= date <= end_date) else 0
            actual_value = existing_row[date_str]
            
            if expected_value != actual_value:
                timeline_changed = True
                break
        
        if timeline_changed:
            additional_changed.append(idx)
    
    if additional_changed:
        print(f"  ⚠ Found {len(additional_changed)} rows with date changes!")
        additional_changed_df = df_pd.loc[additional_changed].copy()
        display(additional_changed_df)
        
        # Merge with changed_rows_df
        changed_rows_df = pd.concat([changed_rows_df, additional_changed_df], ignore_index=True)
    else:
        print("  ✓ No date changes detected")

# Update rows_to_process with all changes
rows_to_process = pd.concat([new_rows, changed_rows_df], ignore_index=True) if len(new_rows) > 0 or len(changed_rows_df) > 0 else pd.DataFrame()

print(f"\n{'='*60}")
print(f"FINAL PROCESSING SUMMARY")
print(f"{'='*60}")
print(f"New rows: {len(new_rows)}")
print(f"Changed rows: {len(changed_rows_df)}")
print(f"Total to process: {len(rows_to_process)}")
print(f"{'='*60}")


Performing deep check on 43 rows with matching metadata...
  ✓ No date changes detected

FINAL PROCESSING SUMMARY
New rows: 0
Changed rows: 0
Total to process: 0


In [0]:
if len(rows_to_process) > 0:
    # Create binary timeline for new rows (same logic as Create Binary Scenario Variables notebook)
    # Create date range from 1973-01 to 2026-12
    date_range = pd.date_range(start='1973-01', end='2026-12', freq='MS')
    date_columns = [d.strftime('%Y-%m') for d in date_range]
    
    # Initialize result rows
    result_rows = []
    
    for idx, row in rows_to_process.iterrows():
        # Check if this is an existing row being updated (need to preserve Coefficient and generic_name)
        composite_key = row['Country'] + '||' + row['Description'] + '||' + row['Event Direction']
        is_existing = composite_key in (existing_full['Country'] + '||' + existing_full['Description'] + '||' + existing_full['Event_Direction']).values
        
        if is_existing:
            # Preserve existing Coefficient and generic_name
            existing_row = existing_full[(existing_full['Country'] + '||' + existing_full['Description'] + '||' + existing_full['Event_Direction']) == composite_key].iloc[0]
            coefficient_value = existing_row['Coefficient']
            generic_name_value = existing_row['generic_name']
        else:
            # New row - initialize as None
            coefficient_value = None
            generic_name_value = None
        
        # Create base info for this event
        event_info = {
            'Country': row['Country'],
            'Event_Direction': row['Event Direction'],  # Note: Google Sheet has space
            'Event_Type': row['Event Type'] if 'Event Type' in row and pd.notna(row['Event Type']) else None,  # Add Event Type
            'Description': row['Description'],
            'Coefficient': coefficient_value,  # Preserve existing value or None for new rows
            'generic_name': generic_name_value  # Preserve existing value or None for new rows
        }
        
        # Determine start and end dates
        start_date = pd.Timestamp(year=int(row['Start Year']), month=int(row['Start Month']), day=1)
        
        # If End Year/Month are null, event ends at the end of its start year
        if pd.isna(row['End Year']) or pd.isna(row['End Month']):
            end_date = pd.Timestamp(year=int(row['Start Year']), month=12, day=1)
        else:
            end_date = pd.Timestamp(year=int(row['End Year']), month=int(row['End Month']), day=1)
        
        # Create binary values for each date column
        for date_str in date_columns:
            date = pd.Timestamp(date_str)
            # Event is active (1) if date is between start and end (inclusive)
            event_info[date_str] = 1 if (start_date <= date <= end_date) else 0
        
        result_rows.append(event_info)
    
    # Create dataframe for new rows
    new_binary_timeline = pd.DataFrame(result_rows)
    
    # Rename Event Direction to Event_Direction for consistency
    new_binary_timeline = new_binary_timeline.rename(columns={'Event_Direction': 'Event_Direction'})
    
    # Ensure column order matches existing table: metadata columns first, then date columns
    metadata_cols = ['Country', 'Event_Direction', 'Event_Type', 'Description', 'Coefficient', 'generic_name']
    date_cols = [col for col in new_binary_timeline.columns if col not in metadata_cols]
    ordered_cols = metadata_cols + date_cols
    new_binary_timeline = new_binary_timeline[ordered_cols]
    
    print(f"\nCreated binary timeline for {len(new_binary_timeline)} events (new + changed)")
    display(new_binary_timeline)
else:
    print("\nSkipping binary timeline creation - no rows to process.")


Skipping binary timeline creation - no rows to process.


In [0]:
if len(rows_to_process) > 0:
    # First, delete old versions of changed rows
    if len(changed_rows_df) > 0:
        print(f"\nDeleting {len(changed_rows_df)} existing rows that have changed...")
        
        # Create list of (Country, Description, Event_Direction) tuples to delete
        # IMPORTANT: Must include Event_Direction since events can have both Production and Consumption entries
        delete_conditions = []
        for idx, row in changed_rows_df.iterrows():
            delete_conditions.append(f"(Country = '{row['Country']}' AND Description = '{row['Description']}' AND Event_Direction = '{row['Event Direction']}')")
        
        delete_where = " OR ".join(delete_conditions)
        
        # Use Delta table DELETE operation
        from delta.tables import DeltaTable
        
        delta_table = DeltaTable.forName(spark, 'workspace.gold.exogenous_variables')
        delta_table.delete(delete_where)
        
        print(f"✓ Deleted {len(changed_rows_df)} old row versions")
    
    # Now append all rows (new + updated)
    # Import types for schema definition
    from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
    
    # Get the existing table's columns to ensure schema compatibility
    existing_columns = existing_table.columns
    
    # Align new_binary_timeline columns to match existing table
    # Keep only columns that exist in the existing table
    aligned_columns = [col for col in existing_columns if col in new_binary_timeline.columns]
    new_binary_timeline_aligned = new_binary_timeline[aligned_columns]
    
    # Define explicit schema matching the existing table's column order
    schema_fields = []
    for col in existing_columns:
        if col in ['Country', 'Event_Direction', 'Event_Type', 'Description', 'generic_name']:
            schema_fields.append(StructField(col, StringType(), True))
        elif col == 'Coefficient':
            schema_fields.append(StructField(col, DoubleType(), True))
        else:
            # Date columns are IntegerType
            schema_fields.append(StructField(col, IntegerType(), True))
    
    schema = StructType(schema_fields)
    
    # Create Spark DataFrame with explicit schema
    new_rows_spark = spark.createDataFrame(new_binary_timeline_aligned, schema=schema)
    
    # Append to existing table (mode='append' preserves existing data)
    action = "new" if len(changed_rows_df) == 0 else f"new + {len(changed_rows_df)} updated"
    print(f"\nAppending {new_rows_spark.count()} rows ({action}) to workspace.gold.exogenous_variables...")
    new_rows_spark.write.mode('append').saveAsTable('workspace.gold.exogenous_variables')
    
    # Verify the update
    updated_count = spark.table('workspace.gold.exogenous_variables').count()
    print(f"\n{'='*60}")
    print(f"✓ TABLE UPDATED SUCCESSFULLY")
    print(f"{'='*60}")
    print(f"Previous count: {existing_table.count()}")
    print(f"New rows added: {len(new_rows)}")
    print(f"Rows updated: {len(changed_rows_df)}")
    print(f"Current count: {updated_count}")
    print(f"{'='*60}")
else:
    print("\n✓ No changes to apply. Table is already up to date.")


✓ No changes to apply. Table is already up to date.


In [0]:
# Display final table summary
final_table = spark.table('workspace.gold.exogenous_variables')

print(f"\n{'='*60}")
print(f"FINAL TABLE SUMMARY")
print(f"{'='*60}")
print(f"Total events: {final_table.count()}")
print(f"\nSample of all events (showing metadata columns only):")
display(final_table.select('Country', 'Event_Direction', 'Event_Type', 'Description', 'Coefficient', 'generic_name').orderBy('Country', 'Description'))


FINAL TABLE SUMMARY
Total events: 63

Sample of all events (showing metadata columns only):


Country,Event_Direction,Event_Type,Description,Coefficient,generic_name
Algeria,Baseline,null,Baseline scenario,null,null
Angola,Baseline,null,Baseline scenario,null,null
Brazil,Baseline,null,Baseline scenario,null,null
Canada,Consumption,Export Cut,1973 OPEC Oil Embargo,0.0,Export Cut in Canada
Canada,Production,New Pipeline,Alberta Clipper Pipeline (Enbridge Line 67) completed,-0.016422532,null
Canada,Baseline,null,Baseline scenario,null,null
Canada,Production,Demand Disruption,COVID Shutdowns,-11.111574960026703,Demand Disruption in Canada
Canada,Consumption,Demand Disruption,COVID Shutdowns,-11.111574960026703,Demand Disruption in Canada
Canada,Production,New Pipeline,Enbridge Line 3 Pipeline replacement completed,-0.0060599651,null
Canada,Consumption,New Pipeline,Enbridge Line 6 Pipeline completed,0.0,New Pipeline in Canada


In [0]:
# Add Japan and Singapore baseline rows to exogenous_variables
# Both countries: Baseline scenario with all date columns = 0
# Only insert if they don't already exist

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import pandas as pd

print("Checking for Japan and Singapore baseline rows...\n")

# Get the existing table to understand its schema
exo_table = spark.table('workspace.gold.exogenous_variables')

# Check which countries already exist with Baseline scenario
existing_baselines = spark.sql("""
    SELECT Country 
    FROM workspace.gold.exogenous_variables
    WHERE Country IN ('Japan', 'Singapore')
      AND Event_Direction = 'Baseline'
      AND Description = 'Baseline scenario'
""").toPandas()

existing_countries = set(existing_baselines['Country'].tolist()) if len(existing_baselines) > 0 else set()

print(f"Existing baseline rows found for: {existing_countries if existing_countries else 'None'}")

# Get all date columns
metadata_cols = ['Country', 'Event_Direction', 'Event_Type', 'Description', 'Coefficient', 'generic_name']
date_cols = [col for col in exo_table.columns if col not in metadata_cols]

print(f"Found {len(date_cols)} date columns (from {date_cols[0]} to {date_cols[-1]})\n")

# Create rows only for countries that don't exist yet
countries_to_add = []
if 'Japan' not in existing_countries:
    countries_to_add.append('Japan')
if 'Singapore' not in existing_countries:
    countries_to_add.append('Singapore')

if not countries_to_add:
    print("\n" + "="*60)
    print("⚠ SKIPPED - All baseline rows already exist")
    print("="*60)
    print("Japan and Singapore baseline rows are already in the table.")
    print("No changes made.")
    print("="*60)
else:
    print(f"Will add baseline rows for: {countries_to_add}\n")
    
    # Create the new rows
    new_rows_data = []
    for country in countries_to_add:
        new_rows_data.append({
            'Country': country,
            'Event_Direction': 'Baseline',
            'Event_Type': None,
            'Description': 'Baseline scenario',
            'Coefficient': None,
            'generic_name': None,
            **{date_col: 0 for date_col in date_cols}
        })

    # Convert to pandas DataFrame first
    new_rows_pd = pd.DataFrame(new_rows_data)
    
    # Define schema matching existing table
    schema_fields = []
    for col_name in exo_table.columns:
        if col_name in ['Country', 'Event_Direction', 'Event_Type', 'Description', 'generic_name']:
            schema_fields.append(StructField(col_name, StringType(), True))
        elif col_name == 'Coefficient':
            schema_fields.append(StructField(col_name, DoubleType(), True))
        else:
            # Date columns are IntegerType
            schema_fields.append(StructField(col_name, IntegerType(), True))
    
    schema = StructType(schema_fields)
    
    # Create Spark DataFrame with matching column order and schema
    new_rows_spark = spark.createDataFrame(new_rows_pd, schema=schema)
    
    print("New rows to insert:")
    print("Country\t\tEvent_Direction\tEvent_Type\tDescription\t\tCoefficient\tgeneric_name")
    for row_data in new_rows_data:
        print(f"{row_data['Country']}\t\t{row_data['Event_Direction']}\t{row_data['Event_Type']}\t\t{row_data['Description']}\t{row_data['Coefficient']}\t{row_data['generic_name']}")
    
    print(f"\nAll {len(date_cols)} date columns set to: 0\n")
    
    # Append to table
    print(f"Inserting {new_rows_spark.count()} rows into workspace.gold.exogenous_variables...")
    new_rows_spark.write.mode('append').saveAsTable('workspace.gold.exogenous_variables')
    
    print("\n" + "="*60)
    print("✓ ROWS ADDED SUCCESSFULLY")
    print("="*60)
    
    # Verify
    verify_df = spark.sql("""
        SELECT Country, Event_Direction, Event_Type, Description, Coefficient, generic_name
        FROM workspace.gold.exogenous_variables
        WHERE Country IN ('Japan', 'Singapore')
          AND Event_Direction = 'Baseline'
          AND Description = 'Baseline scenario'
        ORDER BY Country
    """)
    
    print("\nVerification - Baseline rows in table:")
    display(verify_df)
    
    print(f"\nTotal rows in exogenous_variables: {spark.table('workspace.gold.exogenous_variables').count()}")
    print("="*60)

Checking for Japan and Singapore baseline rows...

Existing baseline rows found for: {'Japan', 'Singapore'}
Found 638 date columns (from 1973-01 to 2026-02)


⚠ SKIPPED - All baseline rows already exist
Japan and Singapore baseline rows are already in the table.
No changes made.


In [0]:
# Assign generic_name = 'Event_Type in Country' to TOP scenarios only
# (Top = highest absolute coefficient per Country + Event_Type + Event_Direction)
print("Assigning generic_name to top scenarios only...\n")

# Find the top 1 coefficient for each Country + Event_Type + Event_Direction combination
top_scenarios = spark.sql("""
SELECT Country, Event_Type, Description, Coefficient, Event_Direction
FROM (
    SELECT 
        Country,
        Event_Type,
        Description,
        Coefficient,
        Event_Direction,
        ROW_NUMBER() OVER (PARTITION BY Country, Event_Type, Event_Direction ORDER BY ABS(Coefficient) DESC, Description) as rn
    FROM workspace.gold.exogenous_variables
    WHERE Coefficient IS NOT NULL
)
WHERE rn = 1
ORDER BY Country, Event_Type, Event_Direction
""")

print(f"Found {top_scenarios.count()} top scenarios to assign generic_name\n")

# First, clear any existing generic names
spark.sql("""
    UPDATE workspace.gold.exogenous_variables
    SET generic_name = NULL
""")

print("Cleared existing generic names")

# Convert the top scenarios dataframe to a temporary view for joining
top_scenarios.createOrReplaceTempView("top_scenarios")

# Update generic_name for the top scenarios
# Assign as 'Event_Type in Country' format
spark.sql("""
    UPDATE workspace.gold.exogenous_variables AS ev
    SET generic_name = CONCAT(ev.Event_Type, ' in ', ev.Country)
    WHERE EXISTS (
        SELECT 1 
        FROM top_scenarios ts
        WHERE ts.Country = ev.Country 
          AND ts.Event_Type = ev.Event_Type 
          AND ts.Event_Direction = ev.Event_Direction
          AND ts.Description = ev.Description
          AND ts.Coefficient = ev.Coefficient
    )
""")

print(f"\n✓ Generic names assigned to {top_scenarios.count()} top scenarios")

# Verify the results
result = spark.sql("""
    SELECT Country, Event_Type, Event_Direction, Description, Coefficient, generic_name
    FROM workspace.gold.exogenous_variables
    WHERE generic_name IS NOT NULL
    ORDER BY Country, Event_Type
""")

print("\nScenarios with generic names assigned:")
display(result)

Assigning generic_name to top scenarios only...

Found 29 top scenarios to assign generic_name

Cleared existing generic names

✓ Generic names assigned to 29 top scenarios

Scenarios with generic names assigned:


Country,Event_Type,Event_Direction,Description,Coefficient,generic_name
Canada,Demand Disruption,Production,COVID Shutdowns,-11.111574960026703,Demand Disruption in Canada
Canada,Demand Disruption,Consumption,COVID Shutdowns,-11.111574960026703,Demand Disruption in Canada
Canada,Export Cut,Consumption,1973 OPEC Oil Embargo,0.0,Export Cut in Canada
Canada,Localized Disruption,Production,Fort McMurray Wildfire,-0.112178,Localized Disruption in Canada
Canada,New Pipeline,Consumption,Enbridge Line 6 Pipeline completed,0.0,New Pipeline in Canada
Canada,New Pipeline,Production,Express Pipeline completed to Casper,0.041399009,New Pipeline in Canada
China,Demand Disruption,Consumption,COVID Shutdowns,134.32126607544419,Demand Disruption in China
Iran,Export Cut,Production,1973 OPEC Oil Embargo,67.533830298778,Export Cut in Iran
Iran,Strike,Production,Iranian Revolution,-736.0047090970448,Strike in Iran
Iran,War,Production,Iran-Iraq War,-163.14283243045497,War in Iran


In [0]:
# Load the exogenous_variables table
exo_table = spark.table('workspace.gold.exogenous_variables')

print(f"Original table has {exo_table.count()} rows")
print(f"Columns: {len(exo_table.columns)}")

# Identify metadata columns vs date columns
metadata_cols = ['Country', 'Event_Direction', 'Event_Type', 'Description', 'Coefficient', 'generic_name']
date_cols = [col for col in exo_table.columns if col not in metadata_cols]

print(f"\nMetadata columns: {len(metadata_cols)}")
print(f"Date columns to unpivot: {len(date_cols)} (from {date_cols[0]} to {date_cols[-1]})")

# Unpivot using stack() - creates one row per date per event
from pyspark.sql.functions import expr, col, to_date

# Build the stack expression: stack(n, 'col1', col1, 'col2', col2, ...)
stack_expr = f"stack({len(date_cols)}, {', '.join([f"'{c}', `{c}`" for c in date_cols])}) as (Date, Value)"

print(f"\nUnpivoting {len(date_cols)} date columns...")

# Apply stack to unpivot
exo_long = exo_table.select(
    *metadata_cols,
    expr(stack_expr)
)

# Convert Date string (YYYY-MM) to actual date type (first day of month)
from pyspark.sql.functions import concat, lit
exo_long = exo_long.withColumn('Date', to_date(concat(col('Date'), lit('-01')), 'yyyy-MM-dd'))

# Rename Value to Binary_Value for clarity
exo_long = exo_long.withColumnRenamed('Value', 'Binary_Value')

print(f"\n✓ Unpivoted table created with {exo_long.count()} rows")

# Show sample
print("\nSample of unpivoted data:")
display(exo_long.orderBy('Country', 'Description', 'Date').limit(20))

# Save to new table
print(f"\nSaving to workspace.gold.exogenous_variables_long...")
# Use mergeSchema to handle schema evolution (Event_Type column was added)
exo_long.write.mode('overwrite').option('mergeSchema', 'true').saveAsTable('workspace.gold.exogenous_variables_long')

print("\n" + "="*60)
print("✓ TABLE CREATED SUCCESSFULLY")
print("="*60)
print(f"Table: workspace.gold.exogenous_variables_long")
print(f"Total rows: {exo_long.count():,}")
print(f"Date range: {date_cols[0]} to {date_cols[-1]}")
print("\nSchema:")
print("  - Country (string)")
print("  - Event_Direction (string)")
print("  - Event_Type (string)")
print("  - Description (string)")
print("  - Coefficient (double)")
print("  - generic_name (string)")
print("  - Date (date)")
print("  - Binary_Value (int)")
print("="*60)

Original table has 63 rows
Columns: 644

Metadata columns: 6
Date columns to unpivot: 638 (from 1973-01 to 2026-02)

Unpivoting 638 date columns...

✓ Unpivoted table created with 40194 rows

Sample of unpivoted data:


Country,Event_Direction,Event_Type,Description,Coefficient,generic_name,Date,Binary_Value
Algeria,Baseline,null,Baseline scenario,null,null,1973-01-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-02-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-03-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-04-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-05-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-06-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-07-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-08-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-09-01,0
Algeria,Baseline,null,Baseline scenario,null,null,1973-10-01,0



Saving to workspace.gold.exogenous_variables_long...

✓ TABLE CREATED SUCCESSFULLY
Table: workspace.gold.exogenous_variables_long
Total rows: 40,194
Date range: 1973-01 to 2026-02

Schema:
  - Country (string)
  - Event_Direction (string)
  - Event_Type (string)
  - Description (string)
  - Coefficient (double)
  - generic_name (string)
  - Date (date)
  - Binary_Value (int)
